In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
jax.config.update('jax_platform_name', 'cpu')
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results,
    load_modeling_result,
    provide_modeling_results_overview,
    load_excitation_result,
    load_testing_sequences,
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result, evaluate_model_prediction, evaluate_model_rollout,
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

In [ ]:
results = list_modeling_results(
    system_name=None,
    model_type="gp_transition",
    status="completed",
    exp_name="submission_nov",
    excitation_objective=None,
)
print(len(results))
results

In [ ]:
provide_modeling_results_overview(results)

---

In [ ]:
import paramax
from admpe.data_management import load_validation_sequences

jax.config.update("jax_enable_x64", True)

modeling_result = load_modeling_result(results[0]["path"], dtype=jnp.float64)
gp = modeling_result["model"]
metadata = modeling_result["metadata"]

print(
    metadata["system_name"],
    metadata["excitation_objective"],
    f"N={gp.N}",
    f"dtype={gp.alpha.dtype}",
    f"runtime={metadata['elapsed_seconds'] / 60:.1f} min",
)
print("Finite weights:", bool(jnp.all(jnp.isfinite(gp.alpha))))

In [ ]:
histories = metadata["validation"]["gp_optimizer_histories"]
max_iters = metadata["params"]["model_trainer_params"]["max_iters"]

fig, ax = plt.subplots(figsize=(9, 4))
rows = []

for output, values in enumerate(histories):
    h = np.asarray(values, dtype=float)
    steps = len(h) - 1
    tail = min(5, steps)

    rows.append({
        "output": output,
        "iterations": steps,
        "reached_cap": steps >= max_iters,
        "initial_NLL": h[0],
        "final_NLL": h[-1],
        "last_step_improvement": h[-2] - h[-1] if steps else np.nan,
        "mean_last_5_improvement": (
            (h[-tail - 1] - h[-1]) / tail if tail else np.nan
        ),
    })
    ax.plot(np.arange(len(h)), h / gp.N, marker=".", label=f"output {output}")

ax.set(
    xlabel="Optimizer iteration",
    ylabel="Negative log marginal likelihood / training point",
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

display(pd.DataFrame(rows).set_index("output"))

In [ ]:
rows = []

for output, posterior in enumerate(gp.posteriors):
    fitted = paramax.unwrap(posterior)
    kernel = fitted.prior.kernel

    rows.append({
        "output": output,
        "variance": float(np.asarray(kernel.variance)),
        "observation_stddev": float(np.asarray(fitted.likelihood.obs_stddev)),
        **{
            f"lengthscale_{i}": value
            for i, value in enumerate(np.asarray(kernel.lengthscale).ravel())
        },
    })

display(pd.DataFrame(rows).set_index("output"))

In [ ]:
validation_data = load_validation_sequences(metadata["system_name"])
rng = np.random.default_rng(42)

@eqx.filter_jit
def predict_one_step(model, observations, actions):
    return jax.vmap(model.step, in_axes=(0, 0, None))(
        observations, actions, 1.0
    )

rows = []

for signal, data in validation_data.items():
    observations = data["observations"]
    actions = data["actions"]

    sequence_idx = rng.integers(observations.shape[0], size=128)
    time_idx = rng.integers(actions.shape[1], size=128)

    x = jnp.asarray(observations[sequence_idx, time_idx], dtype=jnp.float64)
    u = jnp.asarray(actions[sequence_idx, time_idx], dtype=jnp.float64)
    y = jnp.asarray(observations[sequence_idx, time_idx + 1], dtype=jnp.float64)

    predictions = jnp.concatenate([
        predict_one_step(gp, x[start:start + 16], u[start:start + 16])
        for start in range(0, len(x), 16)
    ])

    # Featurization handles angle periodicity consistently with model evaluation.
    errors = np.asarray(gp.featurize(predictions) - gp.featurize(y))
    persistence_errors = np.asarray(gp.featurize(x) - gp.featurize(y))

    mse = np.mean(errors**2)
    persistence_mse = np.mean(persistence_errors**2)

    rows.append({
        "signal": signal,
        "GP_RMSE": np.sqrt(mse),
        "persistence_RMSE": np.sqrt(persistence_mse),
        "skill_vs_persistence": (
            1 - mse / persistence_mse if persistence_mse > 0 else np.nan
        ),
        "nonfinite_prediction_fraction": float(
            jnp.mean(~jnp.all(jnp.isfinite(predictions), axis=-1))
        ),
    })

display(pd.DataFrame(rows).set_index("signal"))

In [ ]:
wrapped_model, wrapped_env = get_wrapped_model_and_env(modeling_result)

signal = "aprbs" if "aprbs" in validation_data else next(iter(validation_data))
data = validation_data[signal]

indices = rng.choice(
    data["observations"].shape[0],
    size=min(4, data["observations"].shape[0]),
    replace=False,
)
horizon = min(100, data["actions"].shape[1])

rollout = evaluate_model_rollout(
    wrapped_model,
    observations=jnp.asarray(
        data["observations"][indices, :horizon + 1], dtype=jnp.float64
    ),
    actions=jnp.asarray(
        data["actions"][indices, :horizon], dtype=jnp.float64
    ),
    horizons=[h for h in (1, 10, 50, 100) if h <= horizon],
)

display(pd.DataFrame([
    {
        "horizon": h,
        "averaged_RMSE": float(scores["averaged"]["rmse"]),
        "terminal_RMSE": float(scores["terminal"]["rmse"]),
    }
    for h, scores in rollout["metrics"]["horizons"].items()
]).set_index("horizon"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(
    np.arange(1, horizon + 1),
    np.asarray(rollout["metrics"]["rmse_by_step"]),
)
ax.set(xlabel="Prediction step", ylabel="RMSE", title=f"{signal}: free rollout")
ax.grid(alpha=0.3)
plt.show()